# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, using the [FAIR² croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"  Published: {dataset.metadata.datePublished}")
print(f"  License: {dataset.metadata.license}")
print("  Keywords:", getattr(dataset.metadata, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print details of the dataset's record sets (tables), including their `@id` and field columns (each with their `@id`).

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.recordSet
print(f"Number of Record Sets: {len(record_sets)}\n")
rs_ids = []
for rs in record_sets:
    print(f"RecordSet Name: {getattr(rs, 'name', '<no name>')}")
    print(f"  @id: {rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', '<no id>')}")")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    rs_id = rs["@id"] if isinstance(rs, dict) else getattr(rs, "@id", "<no id>")
    rs_ids.append(rs_id)
    print(f"  Fields:")
    fields = getattr(rs, 'field', getattr(rs, 'fields', []))
    if isinstance(fields, dict):
        fields = [fields]
    for field in (fields or []):
        # field may be croissant.Field or dict
        fid = field["@id"] if isinstance(field, dict) else getattr(field, "@id", "<no id>")
        name = field["name"] if isinstance(field, dict) and 'name' in field else getattr(field, "name", "<no name>")
        dtype = field["dataType"] if isinstance(field, dict) and 'dataType' in field else getattr(field, "dataType", "-")
        print(f"    - {name} (@id: {fid}, dataType: {dtype})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all record sets into DataFrames for further analysis. You can use the record set and field `@id`s printed above to target specific data.

In [ ]:
# Set up a dictionary of DataFrames by record set @id
dataframes = {}

for record_set in record_sets:
    rs_id = record_set["@id"] if isinstance(record_set, dict) else getattr(record_set, "@id", None)
    print(f"Loading records for RecordSet: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[rs_id] = df
            print(f"Shape: {df.shape}. Columns: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print('No records.')
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**The following is an example using the first populated record set. Modify the variable values as appropriate for your analysis.**

In [ ]:
# Identify a record set with data
if len(dataframes) == 0:
    print("No data extracted to analyze. Please check data extraction above.")
else:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Analysis for record set: {first_rs_id}")
    print(f"Available columns: {df.columns.tolist()}")
    
    # Pick a likely numeric field by heuristic (e.g. field contains 'count', 'abundance', 'coverage', or ends with a number type)
    import re
    numeric_candidates = [c for c in df.columns if re.search(r'count|abundance|mw|coverage|peptide|score|amount|intensity|area|quant|pI|MZ', c, re.IGNORECASE)]
    if not numeric_candidates:
        # Fallback: try any column with float/integer dtype
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_candidates.append(c)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Analyzing numeric field: {numeric_field}")

        # Choose a threshold (example: mean or median)
        try:
            threshold = df[numeric_field].dropna().astype(float).median()
        except Exception:
            print("Selected column could not be converted to float. Please select a numeric column.")
            threshold = None

        if threshold is not None:
            filtered_df = df[df[numeric_field].astype(float) > threshold]
            print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold}:")
            print(filtered_df.head())
            # Normalize
            filtered_df = filtered_df.copy()
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
            ) / filtered_df[numeric_field].astype(float).std()
            print(f"\nNormalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a text/categorical field
            string_candidates = [c for c in df.columns if c not in numeric_candidates]
            group_field = None
            for c in string_candidates:
                if df[c].nunique() > 1 and df[c].nunique() < len(df)//2:
                    group_field = c
                    break
            if group_field is not None:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"\nGrouped data by {group_field}:")
                print(grouped_df[[numeric_field, f"{numeric_field}_normalized"]].head())
            else:
                print("No suitable categorical field found for grouping.")
        else:
            print('No valid numeric field for filtering.')
    else:
        print("No numeric field found in first record set DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This cell creates histograms and scatterplots for numeric relationships if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    df = dataframes[list(dataframes.keys())[0]]
    # Visualize the numeric field's distribution
    # Using the previous section's logic to pick a field
    numeric_field_plot = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_plot = col
            break
    if numeric_field_plot is None:
        # Try to cast candidate columns to float and plot
        for col in df.columns:
            try:
                df[col+'_float'] = df[col].astype(float)
                numeric_field_plot = col+'_float'
                break
            except Exception:
                continue
    if numeric_field_plot:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_plot].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field_plot}')
        plt.xlabel(numeric_field_plot)
        plt.ylabel('Frequency')
        plt.show()
        
        # If at least 2 numeric columns, plot a scatterplot
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(numeric_cols) >= 2:
            plt.figure(figsize=(6, 5))
            sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.title(f"Scatterplot of {numeric_cols[0]} vs. {numeric_cols[1]}")
            plt.show()
    else:
        print("No numeric fields found for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and process data from the FAIR² dataset, referencing all entities by their `@id`. Customize the analysis by selecting different record sets or fields based on the dataset overview, and consider extending the EDA and visualization sections for more specific research questions or downstream applications.